In [1]:
import pandas as pd
from sklearn.preprocessing import RobustScaler

df = pd.read_csv('../data/creditcard.csv')

# Fit one scaler on both raw columns to preserve a single, stable artifact for inference.
scaler = RobustScaler()
scaled = scaler.fit_transform(df[['Amount', 'Time']])
df['Amount_scaled'] = scaled[:, 0]
df['Time_scaled'] = scaled[:, 1]

df.drop(columns=['Amount', 'Time'], inplace=True)

In [2]:
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # ← preserves the 0.17% fraud ratio in both splits
)

print(f"Train: {X_train.shape}, fraud: {y_train.sum()}")
print(f"Test:  {X_test.shape},  fraud: {y_test.sum()}")

Train: (227845, 30), fraud: 394
Test:  (56962, 30),  fraud: 98


In [3]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)

print(f"After SMOTE — fraud: {y_train_resampled.sum()}, legit: {(y_train_resampled == 0).sum()}")
# Should be balanced ~50/50 now

After SMOTE — fraud: 227451, legit: 227451


In [4]:
import joblib
from pathlib import Path

Path('../data').mkdir(parents=True, exist_ok=True)
Path('../backend/models').mkdir(parents=True, exist_ok=True)

joblib.dump((X_train_resampled, y_train_resampled), '../data/train_smote.joblib')
joblib.dump((X_test, y_test), '../data/test.joblib')
joblib.dump(scaler, '../backend/models/scaler.joblib')

print('Saved: ../data/train_smote.joblib')
print('Saved: ../data/test.joblib')
print('Saved: ../backend/models/scaler.joblib')

Saved: ../data/train_smote.joblib
Saved: ../data/test.joblib
Saved: ../backend/models/scaler.joblib
